In [0]:
"""
01_bronze_ingestion.py

Purpose:
--------
This notebook implements the Bronze layer of the Medallion Architecture.

The notebook performs:
1. Raw CSV ingestion
2. Minimal transformations
3. Column standardization
4. Conversion of all columns to STRING type
5. Storage of raw Delta tables

Execution Order:
----------------
1. 01_bronze_ingestion
2. 02_silver_transformation
3. 03_gold_datamart

Assumptions:
------------
- Source files are provided as CSV format
- Source files are already downloaded locally
- No business transformations are applied in Bronze layer
- Bronze layer stores raw historical data
"""

# --------------------------------------------------
# Import Required Libraries
# --------------------------------------------------

from pyspark.sql.functions import col

# --------------------------------------------------
# Bronze Layer - Raw Data Ingestion
# --------------------------------------------------
# The Bronze layer stores raw ingested data.
# Only minimal standardization is performed.
# No business logic transformations are applied.
# --------------------------------------------------

# --------------------------------------------------
# DDL Statements
# --------------------------------------------------
# Explicit DDLs are included to satisfy
# assignment requirements, but in production I'd choose saveAsTable("bronze_item") already creates the table.
#Newer Databricks Free Edition environments, you usually do not have permission to create a custom catalog so used default for assignment
# --------------------------------------------------
spark.sql("""

CREATE TABLE IF NOT EXISTS bronze_item
USING DELTA

""")

spark.sql("""

CREATE TABLE IF NOT EXISTS bronze_event
USING DELTA

""")

# --------------------------------------------------
# Source File Paths
# --------------------------------------------------
# Input files are downloaded from:
# https://merkle-de-interview-case-study.s3.eu-central-1.amazonaws.com/de/item.csv
#
# https://merkle-de-interview-case-study.s3.eu-central-1.amazonaws.com/de/event.csv
#
# Files are stored locally in the workspace.
# --------------------------------------------------

item_path = "/Workspace/Users/khushbu675971@gmail.com/interview_assignments/input_feeds/item.csv"

event_path = "/Workspace/Users/khushbu675971@gmail.com/interview_assignments/input_feeds/event.csv"

# --------------------------------------------------
# Read Item CSV File
# --------------------------------------------------
# inferSchema=False is used because Bronze
# layer requires raw schema preservation.
# --------------------------------------------------

item_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(item_path)
)

# --------------------------------------------------
# Read Event CSV File
# --------------------------------------------------
# escape option is added because payload
# contains escaped JSON strings.
# --------------------------------------------------

event_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("escape", '"')
    .csv(event_path)
)

# --------------------------------------------------
# Standardize Event Column Names
# --------------------------------------------------
# Special characters such as dots and spaces
# are replaced to improve maintainability
# and avoid Spark column resolution issues.
# --------------------------------------------------

for c in event_df.columns:
    clean_col = (

        c.replace(".", "_")
         .replace(" ", "_")
         .lower()
    )
    event_df = event_df.withColumnRenamed(
        c,
        clean_col
    )

# --------------------------------------------------
# Convert All Columns to STRING
# --------------------------------------------------
# Bronze layer stores all attributes
# as STRING type according to assignment
# requirements.
# --------------------------------------------------

item_df = item_df.select(
    [
        col(c).cast("string").alias(c)
        for c in item_df.columns
    ]
)

event_df = event_df.select(
    [
        col(c).cast("string").alias(c)
        for c in event_df.columns
    ]
)
    
# --------------------------------------------------
# Data Validation Checks
# --------------------------------------------------
# Basic validation checks are performed
# before saving Bronze tables.
# --------------------------------------------------

print("Item Record Count:")

print(item_df.count())

print("Event Record Count:")

print(event_df.count())

print("Item Schema:")

item_df.printSchema()

print("Event Schema:")

event_df.printSchema()

# --------------------------------------------------
# Save Bronze Delta Tables
# --------------------------------------------------
# Raw datasets are stored in Delta format.
# overwrite mode is used for assignment
# simplicity.
# --------------------------------------------------

item_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_item")

event_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_event")

# --------------------------------------------------
# Bronze Layer Completion
# --------------------------------------------------

print("Bronze layer completed successfully.")

print("Tables Created:")

print("- bronze_item")

print("- bronze_event")